# LLM tactical report — free-tier test

Sends match-stat summaries (from `team_pressure_analysis_v11.ipynb`) to an LLM and asks it to write tactical commentary. Starting with **Google Gemini's free tier** via AI Studio — no credit card needed, and it's a solid default for this kind of writing task. Swap the model/provider later (e.g. Groq, OpenRouter) once this pipeline stage is working.

Get a free key at https://aistudio.google.com/apikey (takes ~1 minute, no billing required for the free tier).

In [ ]:
import requests
from getpass import getpass

GEMINI_API_KEY = getpass("Paste your Gemini API key (from https://aistudio.google.com/apikey): ")
GEMINI_MODEL = "gemini-3.6-flash"  # current default flash model as of Sep 2026 -- free tier, fast
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"

## Setup

Plain REST call via `requests` — no SDK dependency, so this runs the same in your local `venv_clean`, on Kaggle, or on Colab without extra installs.

**Note on the model name:** Google retires/renames free-tier model IDs periodically (e.g. `gemini-2.5-flash` is already gone for new users as of this writing). If `GEMINI_MODEL` below 404s, check the current default at https://ai.google.dev/gemini-api/docs/models and swap it in — the rest of the code doesn't need to change.

In [ ]:
def call_gemini(prompt, temperature=0.4, max_output_tokens=1024):
    """Minimal REST call to the Gemini API."""
    headers = {"Content-Type": "application/json"}
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": temperature,
            "maxOutputTokens": max_output_tokens,
        },
    }
    resp = requests.post(
        f"{GEMINI_URL}?key={GEMINI_API_KEY}",
        headers=headers,
        json=payload,
        timeout=60,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Gemini API error {resp.status_code}: {resp.text}")
    data = resp.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

## Test call

Confirms the key + endpoint work before sending anything real.

In [ ]:
test_response = call_gemini("In one sentence, confirm you're working and name the model you are.")
print(test_response)

## Loading match stats from the pressure-analysis notebook

Reads `match_stats_summary.json`, which the new export cell (section 15) at the end of `team_pressure_analysis_v11.ipynb` writes automatically next to your other caches. Run that cell there first (once per match/analysis run) so this notebook always picks up the latest numbers instead of anything being copy-pasted by hand.

In [ ]:
import json
from pathlib import Path
import paths

MATCH_STATS_PATH = Path(paths.PLAYER_FRAME_TABLE_CACHE_PATH).parent / "match_stats_summary.json"

with open(MATCH_STATS_PATH) as f:
    match_stats = json.load(f)

match_stats

In [ ]:
def build_tactical_prompt(stats):
    return f"""You are a football tactics analyst. Write a short tactical report (3-4 paragraphs)
based on the following match data. Focus on what the pressing/defensive-line numbers imply about
each team's approach, and note anything statistically interesting or contradictory.

Match: {stats['match']}
Team names: {stats['team_names']}
Defensive line depth (m from own goal, out of possession): {stats['defensive_line_depth_m']}
PPDA (passes allowed per defensive action -- lower means more intense pressing): {stats['ppda']}
Pressure events detected: {stats['pressure_events']}
Counter-press regain rate (won the ball back within 5s of losing it): {stats['counter_press_regain_rate_pct']}
Regain rate specifically when a fast press (<=5s) was mounted: {stats['regain_rate_when_pressed_fast_pct']}
Stoppages: {stats['stoppages']} (total {stats['total_dead_time_sec']}s dead time, excluded from the stats above)

Write in plain English, no bullet points, as if for a coaching report. If any numbers seem to
contradict each other (e.g. a team presses fast often but it doesn't correlate with winning the
ball back), point that out explicitly rather than glossing over it."""

In [ ]:
prompt = build_tactical_prompt(match_stats)
report = call_gemini(prompt, temperature=0.4, max_output_tokens=800)
print(report)

## Notes

- **Free tier limits** apply per Google Cloud project and change over time — check what's live on your key at https://ai.google.dev/gemini-api/docs/rate-limits or the quota panel in AI Studio, rather than trusting a hardcoded number here.
- **If you hit rate limits while testing**, other free tiers worth trying: Groq (fast, OSS models like Llama, roughly 30 requests/min and 1,000/day as of mid-2026) or OpenRouter's `:free`-suffixed models (roughly 20 requests/min). Swapping providers just means changing `call_gemini`'s URL/payload shape — the prompt-building logic below stays the same.
- Once this is solid, the natural next step is replacing `match_stats` with real output loaded from `team_pressure_analysis_v11.ipynb` (save it to JSON/pickle there, read it here), turning this into the LLM-reporting stage of the full pipeline.